# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    print ("Nieprawidlowo wpisales API-KEY w secrets.")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś dla Ciebie szczególnie trudna?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [ ]:
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój":"twój",
    "moje": "twoje",
    "moją": "twoją",
    "moim": "twoim",
    "moich": "twoich",
    "mam": "masz",
    "mi": "ci",
    "ja": "ty",
    "wiem": "wiesz",
    "ci": "mi",
    "ty": "ja",
    "tobie": "mnie",
    "ciebie": "mnie",
    "twój": "mój",
    "twoja": "moja",
    "jesteś": "jestem",
    "on": "jego",
}

def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s, s) for s in slowa)

reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r"chcę (.*)",
        ["Dlaczego chcesz {0}?", "Co ci daje to, że chcesz {0}?"]),
    (r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Jak to jest być {0}?"]),

    (r"boję się (.*)",
        ["Dlaczego boisz się {0}?", "Od jak dawna boisz się {0}?"]),

    (r"lubię (.*)",
        ["Co najbardziej lubisz w {0}?", "Dlaczego lubisz {0}?"]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)
    return None

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

print(odpowiedz("Czuję się dzisiaj samotny."))


Czy często tak się czujesz?


In [ ]:
# # === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
# import re, random

# def normalizuj(tekst):
#     return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

# odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
#            "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

# def odbij(fragment):
#     return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

# reguly = [
#     (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
#     (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
#     (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
#     (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
#     (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
# ]

# def odpowiedz(wypowiedz):
#     w = normalizuj(wypowiedz)
#     for wzorzec, szablony in reguly:
#         m = re.search(wzorzec, w)
#         if m:
#             return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

# TESTY = [
#     "Potrzebuję, żeby moja rodzina mnie słuchała.",
#     "Czuję się dzisiaj samotny.",
#     "Mój ojciec ciągle mnie krytykuje.",
#     "Nie wiem, o co ci chodzi.",
#     "Wszyscy mnie ignorują.",
#     "Boję się, że mnie zostawią.",
# ]
# print(odpowiedz("Czuję się dzisiaj samotny."))


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To zrozumiałe, że potrzebujesz poczuć się ważny i wysłuchany przez bliskich. Co dokładnie sprawia, że masz wrażenie, iż cię nie słyszą?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia daje Ci poczucie, że robisz wszystko źle. Jakie emocje w Tobie dominuje, gdy słyszysz te słowa?

> Nie wiem, o co ci chodzi.
  ELIZA: Co masz na myśli?
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   To musi być bardzo bolesne uczucie, kiedy czujesz się niewidzialny dla innych. Chcesz opowiedzieć mi o jakiejś konkretn

**Różnice (Zadanie 8):**

1. ELIZA powtarza słowa kluczowe, a LLM odnosi się do całego sensu zdania. ELIZA reaguje na pojedyncze słowa („rodzina”, „samotny”, „krytykuje”), LLM rozumie kontekst i odpowiada bardziej całościowo.
2. ELIZA zadaje prawie zawsze pytania, często bardzo podobne, LLM dodaje nowe informacje, rozwija temat, czasem okazuje empatię.
3. ELIZA nie ocenia emocji, tylko mechanicznie kontynuuje rozmowę, LLM rozpoznaje emocje (smutek, lęk, frustrację) i odnosi się do nich w odpowiedzi.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [ ]:
przyklady_lamiace = [
    "Nie jestem smutny.",
    "Wczoraj powiedziałem ci, że jestem zmęczony.",
    "Ale super, znowu mam problemy.",
    "Ona mnie ignoruje, ale nie wiem dlaczego.",
]

for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Nie jestem smutny.
  ELIZA: Od jak dawna jesteś smutny?
  LLM:   Rozumiem. Co zatem skłoniło Cię do dzisiejszej rozmowy ze mną?

> Wczoraj powiedziałem ci, że jestem zmęczony.
  ELIZA: Od jak dawna jesteś zmęczony?
  LLM:   Rozumiem. Czy dzisiaj to zmęczenie nadal ci towarzyszy?

> Ale super, znowu mam problemy.
  ELIZA: Co masz na myśli?
  LLM:   Słyszę w Twoim głosie sporo ironii. Co takiego się wydarzyło?

> Ona mnie ignoruje, ale nie wiem dlaczego.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   To musi być dla Ciebie bardzo bolesne i dezorientujące. Czy próbowałeś z nią o tym spokojnie porozmawiać?



**Moje przewidywanie (przed uruchomieniem):** Tak

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To z jego strony bardzo bolesne, zwłaszcza że to brat. Jak zazwyczaj na to reagujesz?
LLM: Jako terapeutka nie znam szczegółów z Twojego życia prywatnego. Chciałbyś mi o nim opowiedzieć?


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✔️  | LLM rozumie sens zdania, a nie tylko słowa kluczowe — potrafi zauważyć, że „nie jestem smutny” to zaprzeczenie. |
| ironia | ❌ | ✔️  | LLM analizuje ton wypowiedzi i kontekst, więc rozpoznaje ironię („Ale super, znowu mam problemy”). |
| polska gramatyka / odbicie zaimków | ❌ | ✔️  | LLM poprawnie interpretuje zaimki („ona mnie ignoruje”), bo przetwarza strukturę zdania, a nie tylko pojedyncze słowa. |
| pamięć poprzednich zdań | ❌ | ✔️  | LLM potrafi łączyć bieżącą wypowiedź z wcześniejszym kontekstem, ELIZA działa tylko na jedno zdanie naraz. |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te jego słowa?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Współczuję Ci, kłótnie z rodzeństwem potrafią mocno nadszarpnąć energię. Co dokładnie było powodem tego spięcia?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Chcesz o nim porozmawiać?



In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. 
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Współczuję Ci, kłótnie z rodzeństwem potrafią mocno nadszarpnąć energi
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Chcesz o nim porozmawiać?

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 417


**Odpowiedź (Zadanie 10c):** Model „pamięta” tylko dlatego, że przy każdym kolejnym zapytaniu program ponownie wysyła całą historię rozmowy. Sama pamięć nie jest przechowywana przez model — jest odtwarzana z listy wiadomości. Gdy historia nie zostanie wysłana, model traci cały kontekst rozmowy.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    "programista": (
    "Jesteś doświadczonym programistą Pythona. "
    "Odpowiadaj krótko, jasno i zawsze pokazuj przykład kodu."),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

In [ ]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

**Obserwacje (Zadanie 11):**

a) Zmiana wiadomości systemowej całkowicie zmienia zachowanie modelu. Ten sam model może odpowiadać jak terapeuta, przewodnik czy programista tylko dzięki innym instrukcjom.

b) Przy temperaturze 0 odpowiedzi są prawie identyczne i przewidywalne. Przy temperaturze 1.5 odpowiedzi są bardziej różnorodne i kreatywne, ale mogą być mniej spójne.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM naprawił większość problemów ELIZY, ponieważ analizuje znaczenie całego zdania, a nie tylko słowa kluczowe. Lepiej rozumie negację, kontekst i potrafi odpowiadać bardziej naturalnie. Jednak pamięć nadal jest mechaniczna, ponieważ model pamięta tylko to, co zostanie ponownie przesłane w historii rozmowy.
2. W ELIZIE rola była zapisana w regułach programu i jej zmiana wymagała zmiany kodu. W LLM wystarczy zmienić wiadomość systemową, aby model zachowywał się jak terapeuta, przewodnik czy programista. Dzięki temu ten sam model może pełnić wiele różnych ról.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.